## Example 1: Client configuration, authentication and a local test server

In [ ]:
# Example 1: Client configuration, authentication and a local test server
import json
import os
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from unittest.mock import patch

import openai
from openai import OpenAI

print("Example 1 - OpenAI client configuration and authentication")


class FakeOpenAIServer:
    """A local HTTP server that stands in for the OpenAI REST API.

    `routes` maps (method, path) to a function that takes the recorded request and
    returns (status, headers, body bytes). Every request is kept in `requests`, so
    the examples can check exactly what the SDK sent. No network access is needed.
    """

    def __init__(self):
        self.routes = {}
        self.requests = []
        fake = self

        class Handler(BaseHTTPRequestHandler):
            def handle_request(self):
                length = int(self.headers.get("Content-Length") or 0)
                request = {
                    "method": self.command,
                    "path": self.path.split("?")[0],
                    "headers": {name.lower(): value for name, value in self.headers.items()},
                    "body": self.rfile.read(length),
                }
                fake.requests.append(request)
                route = fake.routes.get((request["method"], request["path"]))
                if route is None:
                    error = {"error": {"message": "no such route", "type": "invalid_request_error"}}
                    status, headers, body = 404, {}, json.dumps(error).encode()
                else:
                    status, headers, body = route(request)
                headers = dict(headers)
                self.send_response(status)
                self.send_header("Content-Type", headers.pop("Content-Type", "application/json"))
                for name, value in headers.items():
                    self.send_header(name, value)
                self.send_header("Content-Length", str(len(body)))
                self.end_headers()
                self.wfile.write(body)

            do_GET = do_POST = do_DELETE = handle_request

            def log_message(self, *args):
                pass

        self.httpd = ThreadingHTTPServer(("127.0.0.1", 0), Handler)
        self.base_url = f"http://127.0.0.1:{self.httpd.server_address[1]}/v1"
        threading.Thread(target=self.httpd.serve_forever, daemon=True).start()

    def json_route(self, method, path, payload, status=200, headers=None):
        body = json.dumps(payload).encode()
        self.routes[(method, path)] = lambda request: (status, dict(headers or {}), body)

    def client(self, **kwargs):
        kwargs.setdefault("max_retries", 0)
        kwargs.setdefault("timeout", 10)
        return OpenAI(api_key="sk-test", base_url=self.base_url, **kwargs)

    def last_json(self):
        return json.loads(self.requests[-1]["body"])

    def close(self):
        self.httpd.shutdown()
        self.httpd.server_close()


try:
    print(f"OpenAI library version: {openai.__version__}")

    # Keep any OPENAI_* settings of the host out of these checks
    clean_env = {name: value for name, value in os.environ.items() if not name.startswith("OPENAI_")}
    with patch.dict(os.environ, clean_env, clear=True):
        client = OpenAI(api_key="sk-test")
        assert client.api_key == "sk-test"
        assert str(client.base_url) == "https://api.openai.com/v1/", client.base_url
        print(f"Default base URL: {client.base_url}")

        # with_options returns a copy with new settings and leaves the original alone
        tuned = client.with_options(timeout=20.0, max_retries=5)
        assert (tuned.timeout, tuned.max_retries) == (20.0, 5), (tuned.timeout, tuned.max_retries)
        assert tuned.max_retries != client.max_retries

        # Without a key anywhere the constructor refuses to build a client
        try:
            OpenAI()
        except openai.OpenAIError as e:
            print(f"Missing API key rejected: {type(e).__name__}")
        else:
            raise AssertionError("OpenAI() without an API key must raise OpenAIError")

        # The key and base URL can come from the environment
        with patch.dict(os.environ, {"OPENAI_API_KEY": "sk-from-env", "OPENAI_BASE_URL": "http://127.0.0.1:9/v1"}):
            env_client = OpenAI()
        assert env_client.api_key == "sk-from-env"
        assert str(env_client.base_url) == "http://127.0.0.1:9/v1/", env_client.base_url

    # A request against the local server carries the key, organization, project and custom headers
    fake = FakeOpenAIServer()
    fake.json_route("GET", "/v1/models", {
        "object": "list",
        "data": [
            {"id": "demo-model-a", "object": "model", "created": 1700000000, "owned_by": "demo"},
            {"id": "demo-model-b", "object": "model", "created": 1700000001, "owned_by": "demo"},
        ],
    })
    org_client = fake.client(organization="org-demo", project="proj-demo", default_headers={"X-Demo-App": "notebook"})
    model_ids = [model.id for model in org_client.models.list()]
    print(f"Models listed by the local server: {model_ids}")
    assert model_ids == ["demo-model-a", "demo-model-b"], model_ids
    sent = fake.requests[-1]
    assert (sent["method"], sent["path"]) == ("GET", "/v1/models"), sent
    assert sent["headers"]["authorization"] == "Bearer sk-test"
    assert sent["headers"]["openai-organization"] == "org-demo"
    assert sent["headers"]["openai-project"] == "proj-demo"
    assert sent["headers"]["x-demo-app"] == "notebook"
    fake.close()

    # Nothing listens on port 9: the request fails with APIConnectionError
    unreachable = OpenAI(api_key="sk-test", base_url="http://127.0.0.1:9/v1", max_retries=0, timeout=2)
    try:
        unreachable.models.list()
    except openai.APIConnectionError as e:
        print(f"Unreachable server: {type(e).__name__}")
    else:
        raise AssertionError("A request to an unreachable server must raise APIConnectionError")

    # The exception hierarchy callers rely on
    assert issubclass(openai.APIError, openai.OpenAIError)
    assert issubclass(openai.APIConnectionError, openai.APIError)
    assert issubclass(openai.APITimeoutError, openai.APIConnectionError)
    for status_error in (openai.BadRequestError, openai.AuthenticationError, openai.RateLimitError, openai.InternalServerError):
        assert issubclass(status_error, openai.APIStatusError), status_error

    print("Example 1 completed successfully")

except Exception as e:
    raise AssertionError(
        f"openai example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: Chat completions, tool calls and streaming

In [ ]:
# Example 2: Chat completions, tool calls and streaming
import json

from openai.types.chat import ChatCompletion

print("Example 2 - Chat completions against the local test server")

try:
    fake = FakeOpenAIServer()
    client = fake.client()

    def chat_payload(message, finish_reason="stop", completion_id="chatcmpl-demo-1"):
        return {
            "id": completion_id,
            "object": "chat.completion",
            "created": 1700000000,
            "model": "demo-chat-model",
            "choices": [{"index": 0, "message": message, "logprobs": None, "finish_reason": finish_reason}],
            "usage": {"prompt_tokens": 12, "completion_tokens": 6, "total_tokens": 18},
        }

    # A plain completion: check the request body and the parsed response
    fake.json_route("POST", "/v1/chat/completions",
                    chat_payload({"role": "assistant", "content": "Python is a programming language."}))
    messages = [
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "What is Python?"},
    ]
    completion = client.chat.completions.create(
        model="demo-chat-model", messages=messages, temperature=0.2, max_tokens=50, seed=7,
    )
    print(f"Reply: {completion.choices[0].message.content}")
    assert isinstance(completion, ChatCompletion), type(completion)
    assert completion.choices[0].message.content == "Python is a programming language."
    assert completion.choices[0].finish_reason == "stop"
    assert completion.usage.total_tokens == 18
    request = fake.requests[-1]
    assert (request["method"], request["path"]) == ("POST", "/v1/chat/completions"), request
    assert request["headers"]["authorization"] == "Bearer sk-test"
    sent = fake.last_json()
    assert sent["model"] == "demo-chat-model" and sent["messages"] == messages, sent
    assert (sent["temperature"], sent["max_tokens"], sent["seed"]) == (0.2, 50, 7), sent

    # Tool calling: the tool schema goes out, the call and its JSON arguments come back
    weather_tool = {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Current weather for a city",
            "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]},
        },
    }
    tool_message = {
        "role": "assistant",
        "content": None,
        "tool_calls": [{
            "id": "call_demo_1",
            "type": "function",
            "function": {"name": "get_weather", "arguments": json.dumps({"city": "Paris"})},
        }],
    }
    fake.json_route("POST", "/v1/chat/completions", chat_payload(tool_message, "tool_calls", "chatcmpl-demo-2"))
    reply = client.chat.completions.create(
        model="demo-chat-model",
        messages=[{"role": "user", "content": "What is the weather in Paris?"}],
        tools=[weather_tool],
        tool_choice="auto",
    )
    call = reply.choices[0].message.tool_calls[0]
    print(f"Tool call: {call.function.name}({call.function.arguments})")
    assert reply.choices[0].finish_reason == "tool_calls"
    assert call.id == "call_demo_1" and call.function.name == "get_weather"
    assert json.loads(call.function.arguments) == {"city": "Paris"}
    assert fake.last_json()["tools"] == [weather_tool] and fake.last_json()["tool_choice"] == "auto"

    # Streaming: the SDK parses server-sent events into chunks
    def stream_route(request):
        deltas = [{"role": "assistant", "content": "Hel"}, {"content": "lo"}, {"content": "!"}, {}]
        events = []
        for i, delta in enumerate(deltas):
            events.append({
                "id": "chatcmpl-demo-3",
                "object": "chat.completion.chunk",
                "created": 1700000000,
                "model": "demo-chat-model",
                "choices": [{"index": 0, "delta": delta, "finish_reason": "stop" if i == len(deltas) - 1 else None}],
            })
        body = "".join(f"data: {json.dumps(event)}\n\n" for event in events) + "data: [DONE]\n\n"
        return 200, {"Content-Type": "text/event-stream"}, body.encode()

    fake.routes[("POST", "/v1/chat/completions")] = stream_route
    stream = client.chat.completions.create(
        model="demo-chat-model", messages=[{"role": "user", "content": "Say hello"}], stream=True,
    )
    chunks = list(stream)
    text = "".join(chunk.choices[0].delta.content or "" for chunk in chunks)
    print(f"Streamed {len(chunks)} chunks: {text!r}")
    assert text == "Hello!", text
    assert len(chunks) == 4 and chunks[-1].choices[0].finish_reason == "stop"
    assert fake.last_json()["stream"] is True
    fake.close()

    print("Example 2 completed successfully")

except Exception as e:
    raise AssertionError(
        f"openai example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: Embeddings and vector similarity

In [ ]:
# Example 3: Embeddings and vector similarity
import base64
import json

import numpy as np

print("Example 3 - Embeddings against the local test server")

try:
    fake = FakeOpenAIServer()
    client = fake.client()

    # Fixed 4-dimensional vectors stand in for model output, one per input text
    vectors = {
        "The cat sat on the mat.": [0.9, 0.1, 0.0, 0.1],
        "A cat is sitting on a rug.": [0.8, 0.2, 0.1, 0.1],
        "Interest rates rose sharply today.": [0.0, 0.1, 0.9, 0.3],
    }

    def embeddings_route(request):
        body = json.loads(request["body"])
        inputs = body["input"] if isinstance(body["input"], list) else [body["input"]]
        data = []
        for i, text in enumerate(inputs):
            vector = vectors[text]
            if body.get("encoding_format") == "base64":
                # The API sends base64 of little-endian float32 values in this format
                vector = base64.b64encode(np.asarray(vector, dtype="<f4").tobytes()).decode()
            data.append({"object": "embedding", "index": i, "embedding": vector})
        payload = {
            "object": "list",
            "data": data,
            "model": body["model"],
            "usage": {"prompt_tokens": 7 * len(inputs), "total_tokens": 7 * len(inputs)},
        }
        return 200, {}, json.dumps(payload).encode()

    fake.routes[("POST", "/v1/embeddings")] = embeddings_route

    # A batch request with explicit float output
    texts = list(vectors)
    response = client.embeddings.create(
        model="demo-embedding-model", input=texts, dimensions=4, encoding_format="float",
    )
    sent = fake.last_json()
    assert fake.requests[-1]["path"] == "/v1/embeddings"
    assert sent["input"] == texts and sent["dimensions"] == 4 and sent["encoding_format"] == "float", sent
    assert [item.index for item in response.data] == [0, 1, 2]
    assert response.usage.total_tokens == 21, response.usage
    for item, text in zip(response.data, texts):
        assert np.allclose(item.embedding, vectors[text]), (text, item.embedding)
    print(f"Embedded {len(response.data)} texts, {len(response.data[0].embedding)} dimensions each")

    # Without encoding_format the SDK chooses the wire format (base64 in current
    # releases) and still returns plain floats
    single = client.embeddings.create(model="demo-embedding-model", input=texts[0])
    assert fake.last_json().get("encoding_format") in (None, "base64"), fake.last_json()
    assert np.allclose(single.data[0].embedding, vectors[texts[0]], atol=1e-6), single.data[0].embedding

    # Cosine similarity between the returned vectors
    matrix = np.array([item.embedding for item in response.data])
    unit = matrix / np.linalg.norm(matrix, axis=1, keepdims=True)
    similarity = unit @ unit.T
    print("Cosine similarity matrix:")
    print(np.round(similarity, 3))
    assert np.allclose(np.diag(similarity), 1.0)
    # The two sentences about cats are each other's nearest neighbour
    assert similarity[0, 1] > 0.95 and similarity[0, 2] < 0.1, similarity
    nearest = [int(np.argsort(row)[-2]) for row in similarity]
    assert nearest[:2] == [1, 0], nearest
    fake.close()

    print("Example 3 completed successfully")

except Exception as e:
    raise AssertionError(
        f"openai example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

## Example 4: Image generation, text to speech and transcription

In [ ]:
# Example 4: Image generation, text to speech and transcription requests
import base64
import io
import json
import wave

print("Example 4 - Images and audio against the local test server")

try:
    fake = FakeOpenAIServer()
    client = fake.client()

    # Image generation with base64 output
    png_bytes = b"\x89PNG\r\n\x1a\n" + bytes(16)
    fake.json_route("POST", "/v1/images/generations", {
        "created": 1700000000,
        "data": [{"b64_json": base64.b64encode(png_bytes).decode()}],
    })
    image = client.images.generate(
        model="demo-image-model",
        prompt="A red apple on a wooden table",
        size="1024x1024",
        n=1,
        response_format="b64_json",
    )
    sent = fake.last_json()
    assert fake.requests[-1]["path"] == "/v1/images/generations"
    assert (sent["model"], sent["prompt"], sent["size"], sent["n"], sent["response_format"]) == (
        "demo-image-model", "A red apple on a wooden table", "1024x1024", 1, "b64_json"), sent
    decoded = base64.b64decode(image.data[0].b64_json)
    assert decoded == png_bytes and decoded.startswith(b"\x89PNG")
    print(f"Image bytes decoded: {len(decoded)}")

    # Text to speech returns raw audio bytes, not JSON
    audio_bytes = b"ID3\x04\x00" + bytes(32)
    fake.routes[("POST", "/v1/audio/speech")] = lambda request: (200, {"Content-Type": "audio/mpeg"}, audio_bytes)
    speech = client.audio.speech.create(
        model="demo-tts-model", voice="alloy", input="Hello there", response_format="mp3",
    )
    assert speech.content == audio_bytes
    sent = fake.last_json()
    assert (sent["model"], sent["voice"], sent["input"], sent["response_format"]) == (
        "demo-tts-model", "alloy", "Hello there", "mp3"), sent
    print(f"Speech bytes received: {len(speech.content)}")

    # Transcription uploads the audio file as multipart form data
    buffer = io.BytesIO()
    with wave.open(buffer, "wb") as wav:
        wav.setnchannels(1)
        wav.setsampwidth(2)
        wav.setframerate(16000)
        wav.writeframes(bytes(3200))  # 0.1 s of silence
    wav_bytes = buffer.getvalue()
    fake.json_route("POST", "/v1/audio/transcriptions", {"text": "hello world"})
    transcript = client.audio.transcriptions.create(
        model="demo-stt-model", file=("clip.wav", wav_bytes, "audio/wav"), language="en",
    )
    assert transcript.text == "hello world", transcript
    request = fake.requests[-1]
    assert request["path"] == "/v1/audio/transcriptions"
    assert request["headers"]["content-type"].startswith("multipart/form-data"), request["headers"]
    assert b'filename="clip.wav"' in request["body"] and wav_bytes in request["body"]
    assert b'name="model"' in request["body"] and b"demo-stt-model" in request["body"]
    print(f"Transcription: {transcript.text!r}")
    fake.close()

    print("Example 4 completed successfully")

except Exception as e:
    raise AssertionError(
        f"openai example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: Async client, error mapping, retries and raw responses

In [ ]:
# Example 5: Async client, error mapping, retries and raw responses
import asyncio
import json
import time

import openai
from openai import AsyncOpenAI

print("Example 5 - Async client, errors, retries and raw responses")

try:
    fake = FakeOpenAIServer()

    def chat_payload(content):
        return {
            "id": "chatcmpl-demo",
            "object": "chat.completion",
            "created": 1700000000,
            "model": "demo-chat-model",
            "choices": [{"index": 0, "message": {"role": "assistant", "content": content}, "logprobs": None, "finish_reason": "stop"}],
            "usage": {"prompt_tokens": 3, "completion_tokens": 1, "total_tokens": 4},
        }

    def echo_route(request):
        prompt = json.loads(request["body"])["messages"][-1]["content"]
        return 200, {"x-request-id": "req_demo"}, json.dumps(chat_payload(prompt.upper())).encode()

    fake.routes[("POST", "/v1/chat/completions")] = echo_route
    question = [{"role": "user", "content": "hi"}]

    # The async client: three requests in flight at once
    async_client = AsyncOpenAI(api_key="sk-test", base_url=fake.base_url, max_retries=0, timeout=10)
    prompts = ["alpha", "beta", "gamma"]
    results = await asyncio.gather(*(
        async_client.chat.completions.create(model="demo-chat-model", messages=[{"role": "user", "content": p}])
        for p in prompts
    ))
    await async_client.close()
    replies = [result.choices[0].message.content for result in results]
    print(f"Async replies: {replies}")
    assert replies == ["ALPHA", "BETA", "GAMMA"], replies

    # with_raw_response exposes the HTTP response next to the parsed object
    client = fake.client()
    raw = client.chat.completions.with_raw_response.create(
        model="demo-chat-model", messages=[{"role": "user", "content": "raw"}],
    )
    assert raw.headers.get("x-request-id") == "req_demo", dict(raw.headers)
    assert raw.parse().choices[0].message.content == "RAW"

    # HTTP error statuses map to typed exceptions
    for status, error_type in [
        (400, openai.BadRequestError),
        (401, openai.AuthenticationError),
        (404, openai.NotFoundError),
        (429, openai.RateLimitError),
        (500, openai.InternalServerError),
    ]:
        fake.json_route("POST", "/v1/chat/completions",
                        {"error": {"message": f"demo error {status}", "type": "demo"}}, status=status)
        try:
            client.chat.completions.create(model="demo-chat-model", messages=question)
        except error_type as e:
            assert e.status_code == status, (status, e.status_code)
            assert f"demo error {status}" in str(e), str(e)
            print(f"  HTTP {status} -> {type(e).__name__}")
        else:
            raise AssertionError(f"HTTP {status} did not raise {error_type.__name__}")

    # Retries: two 503 replies, then success; retry-after-ms keeps the waits short
    attempts = []

    def flaky_route(request):
        attempts.append(time.monotonic())
        if len(attempts) < 3:
            busy = {"error": {"message": "busy", "type": "server_error"}}
            return 503, {"retry-after-ms": "10"}, json.dumps(busy).encode()
        return 200, {}, json.dumps(chat_payload("recovered")).encode()

    fake.routes[("POST", "/v1/chat/completions")] = flaky_route
    retried = fake.client(max_retries=2).chat.completions.create(model="demo-chat-model", messages=question)
    print(f"Succeeded after {len(attempts)} attempts")
    assert retried.choices[0].message.content == "recovered"
    assert len(attempts) == 3, attempts

    # With retries off, the first 503 reaches the caller
    attempts.clear()
    try:
        fake.client(max_retries=0).chat.completions.create(model="demo-chat-model", messages=question)
    except openai.InternalServerError as e:
        assert e.status_code == 503 and len(attempts) == 1, (e.status_code, attempts)
    else:
        raise AssertionError("A 503 with max_retries=0 must raise InternalServerError")
    fake.close()

    print("Example 5 completed successfully")

except Exception as e:
    raise AssertionError(
        f"openai example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e